# DisasterIQ — Kaggle GPU fine-tune

Fine-tune PyTorch xView2 on your pre-built **train_subset** (~1449 pairs).

**Before running:**
1. Upload `disasteriq-train-subset.zip` as a private Kaggle dataset (see `docs/KAGGLE_FINETUNE.md`)
2. **Input** → add dataset `disasteriq-train-subset`
3. **Settings** → **Accelerator** → **GPU T4 x2** (recommended; P100 needs cell 2 cu118 PyTorch)

**After GPU restart:** re-run **cells 1 → 4** before the GPU check (session clears variables).

**Run order:** 1 → 2 → 3 → 4 (CPU) → enable GPU → 5 → 6 → 7

In [ ]:
DATASET_SLUG = "disasteriq-train-subset"
REPO_URL = "https://github.com/AhmadRaza4076/DisasterIQ.git"
REPO_BRANCH = "main"
XVIEW2_URL = "https://github.com/michal2409/xView2.git"

from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_ROOT = WORKING / "DisasterIQ"


def find_train_subset(root: Path) -> Path | None:
    if (root / "images").is_dir() and (root / "targets").is_dir():
        return root
    for child in sorted(root.iterdir()):
        if child.is_dir():
            found = find_train_subset(child)
            if found:
                return found
    return None


dataset_path = find_train_subset(INPUT)
if dataset_path is None:
    print("Available input dirs:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else [])
    raise FileNotFoundError(
        f"Dataset not found. Add Input dataset '{DATASET_SLUG}' with images/ labels/ targets/"
    )
print("Dataset OK:", dataset_path)
print("Images:", len(list((dataset_path / "images").glob("*"))))

In [ ]:
# Default Kaggle PyTorch may not support P100 (sm_60). cu118 wheels work on P100 and T4.
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q albumentations opencv-python-headless pyyaml joblib tqdm timm segmentation-models-pytorch
!pip install -q pytorch-lightning torch-optimizer dllogger shapely fire monai

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/AhmadRaza4076/DisasterIQ.git"
REPO_BRANCH = "main"
XVIEW2_URL = "https://github.com/michal2409/xView2.git"
WORKING = Path("/kaggle/working")
REPO_ROOT = WORKING / "DisasterIQ"

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
os.chdir(WORKING)
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} DisasterIQ

xview2_dir = REPO_ROOT / "ml" / "pytorch-xview2"
if xview2_dir.exists():
    shutil.rmtree(xview2_dir)
!git clone --depth 1 {XVIEW2_URL} {xview2_dir}
print("Repos ready at", REPO_ROOT)

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/DisasterIQ")
if not REPO_ROOT.is_dir():
    raise RuntimeError("DisasterIQ repo missing — run cell 3 (clone) first.")

os.chdir(REPO_ROOT)
!chmod +x ml/finetune/run_kaggle_pipeline.sh ml/finetune/train_localization.sh ml/finetune/train_damage.sh
!bash ml/finetune/run_kaggle_pipeline.sh --prep-only

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Settings → Accelerator → GPU T4 x2 (or P100) → Save, restart, re-run."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
# Fail fast if PyTorch build does not support this GPU (e.g. P100 sm_60 on default torch)
_ = torch.randn(4, 4, device="cuda")
print("CUDA tensor test: OK")

In [ ]:
# Training only — skips index.csv regeneration (use after prep or a failed train attempt)
import os
import shutil
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/DisasterIQ")
os.chdir(REPO_ROOT)

# Apex fallback (if not patched yet)
plt_path = REPO_ROOT / "ml/pytorch-xview2/model/plt.py"
text = plt_path.read_text()
apex_old = "from apex.optimizers import FusedAdam, FusedNovoGrad, FusedSGD"
apex_new = """try:
    from apex.optimizers import FusedAdam, FusedNovoGrad, FusedSGD
except ImportError:
    FusedAdam = torch.optim.Adam
    FusedSGD = torch.optim.SGD
    FusedNovoGrad = torch.optim.Adam"""
if apex_old in text:
    plt_path.write_text(text.replace(apex_old, apex_new))
    print("Patched apex fallback in plt.py")

index_csv = REPO_ROOT / "ml/pytorch-xview2/utils/index.csv"
if not index_csv.is_file() or index_csv.stat().st_size < 10:
    raise FileNotFoundError("index.csv missing — run cell 4 (prep) first")

os.environ["FINETUNE_CONFIG"] = str(REPO_ROOT / "ml/finetune/config_subset_kaggle.yaml")
os.environ["DATA_DIR"] = "/kaggle/working/data/train_subset"
os.environ["XVIEW2_INDEX_CSV"] = str(index_csv)
print("index.csv OK:", index_csv)

!bash ml/finetune/train_localization.sh
!CKPT_PRE=/kaggle/working/results/loc/checkpoints/best.ckpt RESULTS_DIR=/kaggle/working/results/dmg bash ml/finetune/train_damage.sh

ckpt_out = Path("/kaggle/working/damage_best.ckpt")
dmg_ckpt = Path("/kaggle/working/results/dmg/checkpoints/best.ckpt")
if dmg_ckpt.exists():
    shutil.copy(dmg_ckpt, ckpt_out)
    print("Exported", ckpt_out)

In [ ]:
from pathlib import Path

ckpt = Path("/kaggle/working/damage_best.ckpt")
alt = Path("/kaggle/working/results/dmg/checkpoints/best.ckpt")
if not ckpt.exists() and alt.exists():
    import shutil
    shutil.copy(alt, ckpt)

if ckpt.exists():
    print(f"SUCCESS: {ckpt} ({ckpt.stat().st_size / 1e6:.1f} MB)")
    print("Download from Output tab → damage_best.ckpt")
    print("Laptop: copy to ml/checkpoints/damage_best.ckpt")
    print(".env: INFERENCE_MODE=pytorch")
else:
    raise FileNotFoundError("No checkpoint — check training logs above")